# LSM Pricing and ITM Mask Reuse
Simple notebook to demonstrate the American pricer (with diagnostics/mask) and how mc_fd reuses the mask for bumps.

In [7]:
import pathlib, sys
repo_root = pathlib.Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import numpy as np
from engines.monte_carlo import MonteCarloPricing
from greeks.mc_fd import MonteCarloFiniteDifference, BumpConfig

In [8]:
pricer = MonteCarloPricing(S_0=100, X=100, sigma=0.2, T=1.0, r=0.05, num_paths=200, steps=40, seed=42)
cashflows, mask = pricer.american_cashflows(
    pricer.simulate_paths(risk_neutral=True, antithetic=False),
    return_mask=True
)
price = cashflows.mean(); price

np.float64(10.191362338497134)

In [9]:
mc_fd = MonteCarloFiniteDifference(pricer, call=False, style="american", bumps=BumpConfig(S_0=1.0), include_all_paths=False)
delta = mc_fd.delta(); delta

-0.4413750091724009

In [10]:
from models.black_scholes import OptionPrice
from greeks.bs_fd import BlackScholesFiniteDifference
euro_pricer = MonteCarloPricing(S_0=100, X=100, sigma=0.2, T=1.0, r=0.05, num_paths=100000, steps=64, seed=123)
euro_fd_mc = MonteCarloFiniteDifference(euro_pricer, call=True, style="european", bumps=BumpConfig(S_0=0.5))
delta_mc = euro_fd_mc.delta()
bs_fd = BlackScholesFiniteDifference(S_0=100, X=100, r=0.05, sigma=0.2, T=1.0, call=True, h_S=0.5)
delta_bs = bs_fd.delta()
delta_mc, delta_bs

(0.6383730902668194, 0.6368091553002699)

# LSM vs Binomial (American, no variance reduction)
Compare American LSM (plain Monte Carlo) against a binomial tree.
Calls should line up with European Black-Scholes; puts should match binomial.



In [15]:
from models.binomial_method import BinomialPricing
from models.black_scholes import OptionPrice
import pandas as pd

S0, K, r, T = 100.0, 100.0, 0.05, 1.0
sigmas = [0.15, 0.2, 0.25, 0.3]
steps_tree = 200
steps_mc = steps_tree
paths = 100_000
seed = 123

rows = []

for sigma in sigmas:
    pricer = MonteCarloPricing(
        S_0=S0, X=K, sigma=sigma, T=T, r=r,
        num_paths=paths, steps=steps_mc, seed=seed
    )

    # Standard LSM: no variance reduction, ITM-only regression mask
    lsm_call, lsm_call_se = pricer.american(
        call=True,
        basis_fn="laguerre",
        antithetic=False,
        include_all_paths=False,
        mask_tolerance=0.0,
    )
    lsm_put, lsm_put_se = pricer.american(
        call=False,
        basis_fn="laguerre",
        antithetic=False,
        include_all_paths=False,
        mask_tolerance=0.0,
    )

    binom = BinomialPricing(S_0=S0, K=K, r=r, sigma=sigma, T=T, steps=steps_tree)
    binom_call = binom.american(call=True)
    binom_put = binom.american(call=False)

    bs = OptionPrice(S0, K, r, sigma, T)
    bs_call = bs.call()
    bs_put = bs.put()

    rows.extend([
        {
            "Sigma": sigma,
            "Option": "American Call",
            "Model": "LSM",
            "Price": lsm_call,
            "StdErr": lsm_call_se,
            "EstimatorVar": lsm_call_se ** 2,
        },
        {
            "Sigma": sigma,
            "Option": "American Call",
            "Model": "Binomial",
            "Price": binom_call,
            "StdErr": None,
            "EstimatorVar": None,
        },
        {
            "Sigma": sigma,
            "Option": "American Call",
            "Model": "Black-Scholes (Euro)",
            "Price": bs_call,
            "StdErr": None,
            "EstimatorVar": None,
        },
        {
            "Sigma": sigma,
            "Option": "American Put",
            "Model": "LSM",
            "Price": lsm_put,
            "StdErr": lsm_put_se,
            "EstimatorVar": lsm_put_se ** 2,
        },
        {
            "Sigma": sigma,
            "Option": "American Put",
            "Model": "Binomial",
            "Price": binom_put,
            "StdErr": None,
            "EstimatorVar": None,
        },
        {
            "Sigma": sigma,
            "Option": "American Put",
            "Model": "Black-Scholes (Euro)",
            "Price": bs_put,
            "StdErr": None,
            "EstimatorVar": None,
        },
    ])


df = pd.DataFrame(rows)

# Add comparison columns per sigma/option group
for sigma in sigmas:
    call_mask = (df["Sigma"] == sigma) & (df["Option"] == "American Call")
    put_mask = (df["Sigma"] == sigma) & (df["Option"] == "American Put")

    binom_call_price = df.loc[call_mask & (df["Model"] == "Binomial"), "Price"].iloc[0]
    binom_put_price = df.loc[put_mask & (df["Model"] == "Binomial"), "Price"].iloc[0]
    bs_call_price = df.loc[call_mask & (df["Model"] == "Black-Scholes (Euro)"), "Price"].iloc[0]
    bs_put_price = df.loc[put_mask & (df["Model"] == "Black-Scholes (Euro)"), "Price"].iloc[0]

    df.loc[call_mask, "Diff vs Binomial"] = df.loc[call_mask, "Price"] - binom_call_price
    df.loc[put_mask, "Diff vs Binomial"] = df.loc[put_mask, "Price"] - binom_put_price
    df.loc[call_mask, "Diff vs BS(E)"] = df.loc[call_mask, "Price"] - bs_call_price
    df.loc[put_mask, "Diff vs BS(E)"] = df.loc[put_mask, "Price"] - bs_put_price

# Pretty display
pd.options.display.float_format = "{:.6f}".format

df



,Sigma,Option,Model,Price,StdErr,EstimatorVar,Diff vs Binomial,Diff vs BS(E)
0,0.150000,American Call,LSM,8.637426,0.035451,0.001257,0.053423,0.045768
1,0.150000,American Call,Binomial,8.584004,NaN,NaN,0.000000,-0.007655
2,0.150000,American Call,Black-Scholes (Euro),8.591658,NaN,NaN,0.007655,0.000000
3,0.150000,American Put,LSM,4.222352,0.016196,0.000262,-0.007403,0.507751
4,0.150000,American Put,Binomial,4.229755,NaN,NaN,0.000000,0.515154
5,0.150000,American Put,Black-Scholes (Euro),3.714601,NaN,NaN,-0.515154,0.000000
6,0.200000,American Call,LSM,10.516746,0.046811,0.002191,0.076155,0.066163
7,0.200000,American Call,Binomial,10.440591,NaN,NaN,0.000000,-0.009992
8,0.200000,American Call,Black-Scholes (Euro),10.450584,NaN,NaN,0.009992,0.000000
9,0.200000,American Put,LSM,6.087366,0.022634,0.000512,0.000984,0.513840
